In [1]:
!pip install langchain_huggingface
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace # imporing required libraryes

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/__init__.py", line 114, in create_command
    module = importlib.import_module(module_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unloc

In [ ]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'API key'

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    huggingfacehub_api_token=os.environ['HUGGINGFACEHUB_API_TOKEN'],
    temperature=0.2
)

# Some wrappers need the token explicitly passed if environment sync is slow
model = ChatHuggingFace(llm=llm, huggingfacehub_api_token=os.environ['HUGGINGFACEHUB_API_TOKEN'])

In [3]:
!pip install langchain_core
!pip install langchain_community

In [6]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 107.5 MB/s eta 0:00:00


In [7]:
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun

In [8]:
search_tool = DuckDuckGoSearchRun()

In [9]:
@tool
def weather(city : str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=3b9e9349523ae9c012ab8de9ef17a9c0&query={city}'
  response = requests.get(url)
  return response.json()

In [12]:
weather.invoke('Karimnagar') # checking if tool works, and it does work

{'request': {'type': 'City',
  'query': 'Karimnagar, India',
  'language': 'en',
  'unit': 'm'},
 'location': {'name': 'Karimnagar',
  'country': 'India',
  'region': 'Telangana',
  'lat': '18.433',
  'lon': '79.150',
  'timezone_id': 'Asia/Kolkata',
  'localtime': '2026-07-27 23:17',
  'localtime_epoch': 1785194220,
  'utc_offset': '5.50'},
 'current': {'observation_time': '05:47 PM',
  'temperature': 25,
  'weather_code': 263,
  'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0025_light_rain_showers_night.png'],
  'weather_descriptions': ['Patchy light drizzle'],
  'astro': {'sunrise': '05:50 AM',
   'sunset': '06:50 PM',
   'moonrise': '05:22 PM',
   'moonset': '03:33 AM',
   'moon_phase': 'Waxing Gibbous',
   'moon_illumination': 94},
  'air_quality': {'co': '195',
   'no2': '7.2',
   'o3': '54',
   'so2': '2.3',
   'pm2_5': '13.6',
   'pm10': '22',
   'us-epa-index': '1',
   'gb-defra-index': '1'},
  'wind_speed': 14,
  'wind_degree': 227,
  

In [26]:
pip install langchain-classic

In [27]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic import hub

In [31]:
from langchain_core.prompts import PromptTemplate

template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

In [42]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm = model,
    tools = [search_tool,weather],
    prompt = prompt
)

In [43]:
# Step 4: Wrap it with AgentExecutor
agent_executer = AgentExecutor(
    agent = agent,
    tools = [search_tool,weather],
    verbose=True
)

In [ ]:
#step 5 invoke
response = agent_executer.invoke({'input':'find tha capital of Telangan and its current temparature and weather details'})
print(response)
# this returns thinking of AI model and actions it took while thinking/reasoning and finally genrating an output

In [ ]:
response['output'] # the final output generated